In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
df = pd.read_csv("clean_products.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(1351, 23)
['product_id', 'product_name', 'category_path', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'img_link', 'product_link', 'cat_level_1', 'cat_level_2', 'cat_level_3', 'cat_level_4', 'cat_level_5', 'cat_level_6', 'cat_level_7', 'product_name_clean', 'about_product_clean', 'category_text', 'clean_text_no_category', 'clean_text_with_category']


,product_id,product_name,category_path,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,product_link,cat_level_1,cat_level_2,cat_level_3,cat_level_4,cat_level_5,cat_level_6,cat_level_7,product_name_clean,about_product_clean,category_text,clean_text_no_category,clean_text_with_category
0,B002PD61Y4,D-Link DWA-131 300 Mbps Wireless Nano USB Adapter (Black),Computers&Accessories|NetworkingDevices|NetworkAdapters|WirelessUSBAdapters,507.0,1208.0,58.0,4.1,8131.0,Connects your computer to a high-speed wireless network|Supports WPA/WPA2 wireless encryption to help prevent outside intrusion and protect your personal information from being exposed.|3 Year Brand Warranty|2.4Ghz frequency band (300mbps)|2 fixed internal Wi-Fi antenna|N300 MIMO Wi-Fi USB Adapter|This is a plug and play device with generic configuration|Compatible with all CCTV DVRs,https://m.media-amazon.com/images/I/31+NwZ8gb1L._SX300_SY300_.jpg,https://www.amazon.in/D-Link-DWA-131-Wireless-Adapter-Black/dp/B002PD61Y4/ref=sr_1_50?qid=1672909126&s=electronics&sr=1-50,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,d link dwa 131 300 mbps wireless nano usb adapter black,connects your computer to a high speed wireless network supports wpa wpa2 wireless encryption to help prevent outside intrusion and protect your personal information from being exposed 3 year brand warranty 2 4ghz frequency band 300mbps 2 fixed internal wi fi antenna n300 mimo wi fi usb adapter this is a plug and play device with generic configuration compatible with all cctv dvrs,computers and accessories networkingdevices networkadapters wirelessusbadapters,d link dwa 131 300 mbps wireless nano usb adapter black connects your computer to a high speed wireless network supports wpa wpa2 wireless encryption to help prevent outside intrusion and protect your personal information from being exposed 3 year brand warranty 2 4ghz frequency band 300mbps 2 fixed internal wi fi antenna n300 mimo wi fi usb adapter this is a plug and play device with generic configuration compatible with all cctv dvrs,d link dwa 131 300 mbps wireless nano usb adapter black connects your computer to a high speed wireless network supports wpa wpa2 wireless encryption to help prevent outside intrusion and protect your personal information from being exposed 3 year brand warranty 2 4ghz frequency band 300mbps 2 fixed internal wi fi antenna n300 mimo wi fi usb adapter this is a plug and play device with generic configuration compatible with all cctv dvrs computers and accessories networkingdevices networkadapters wirelessusbadapters
1,B002SZEOLG,"TP-Link Nano USB WiFi Dongle 150Mbps High Gain Wireless Network Wi-Fi Adapter for PC Desktop and Laptops, Supports Windows 10/8.1/8/7/XP, Linux, Mac OS X (TL-WN722N)",Computers&Accessories|NetworkingDevices|NetworkAdapters|WirelessUSBAdapters,749.0,1339.0,44.0,4.2,179692.0,"150 Mbps Wi-Fi —— Exceptional wireless speed up to 150Mbps brings best experience for video streaming or internet calls|Easy Set up —— Easy wireless security encryption at a push of the WPS button|Antenna —— 4dBi detachable Omni Directional antenna, remarkably strengthen signal power of the USB adapter|Compatibility —— Windows 11/10/8.1/8/7/XP, Mac OS 10.15 and earlier, Linux|Interface —— USB 2.0|In an unlikely case of product quality related issue, we may ask you to reach out to brand’s customer service support and seek resolution. We will require brand proof of issue to process replacement request.",https://m.media-amazon.com/images/I/31Wb+A3VVdL._SY300_SX300_.jpg,https://www.amazon.in/TP-Link-TL-WN722N-150Mbps-Wireless-Adapter/dp/B002SZEOLG/ref=sr_1_162?qid=1672909131&s=electronics&sr=1-162,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,tp link nano usb wifi dongle 150mbps high gain wireless network wi fi adapter for pc desktop and laptops supports windows 10 8 1 8 7 xp linux ma

# Recpmmender không cat 

In [75]:
cols = [
    "product_id",
    "product_name",
    "category_path",
    "clean_text_no_category",
    "clean_text_with_category",
    "rating",
    "rating_count",
    "img_link",
    "product_link",
    "cat_level_1",
    "cat_level_2",
    "cat_level_3",
    "cat_level_4",
    "cat_level_5"
]

df = df[cols].copy()

df["clean_text_no_category"] = df["clean_text_no_category"].fillna("")
df["clean_text_with_category"] = df["clean_text_with_category"].fillna("")

In [76]:
vectorizer_no_cat = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix_no_cat = vectorizer_no_cat.fit_transform(df["clean_text_no_category"])

print(tfidf_matrix_no_cat.shape)

(1351, 5000)


In [77]:
similarity_no_cat = cosine_similarity(tfidf_matrix_no_cat, tfidf_matrix_no_cat)

print(similarity_no_cat.shape)

(1351, 1351)


In [78]:
product_idx_map = pd.Series(df.index, index=df["product_id"]).to_dict()

print(list(product_idx_map.items())[:5])

[('B002PD61Y4', 0), ('B002SZEOLG', 1), ('B003B00484', 2), ('B003L62T7W', 3), ('B004IO5BMQ', 4)]


In [79]:
def recommend_no_category(product_id, top_k=10):
    if product_id not in product_idx_map:
        return pd.DataFrame()

    idx = product_idx_map[product_id]

    sim_scores = list(enumerate(similarity_no_cat[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # bỏ chính nó
    sim_scores = sim_scores[1:top_k+1]

    product_indices = [x[0] for x in sim_scores]
    scores = [x[1] for x in sim_scores]

    result = df.iloc[product_indices][[
        "product_id",
        "product_name",
        "category_path",
        "rating",
        "rating_count",
        "img_link",
        "product_link"
    ]].copy()

    result["similarity_score"] = scores
    return result

In [80]:
pd.set_option("display.max_columns", None)       # hiện tất cả cột
pd.set_option("display.max_colwidth", None)      # hiện đầy đủ nội dung trong ô
pd.set_option("display.width", None)             # tự mở rộng chiều rộng hiển thị
pd.set_option("display.max_rows", 100)           # hiện tối đa 100 dòng
pd.set_option("display.expand_frame_repr", False)
sample_id = df["product_id"].iloc[100]

print("Sản phẩm đầu vào:")
print(df[df["product_id"] == sample_id][["product_name","product_link"]])

print("\nTop gợi ý không dùng category:")
recommend_no_category(sample_id, top_k=10)

Sản phẩm đầu vào:
                                                                                                                                                                                      product_name                                                                                                                          product_link
100  Storite USB Extension Cable USB 3.0 Male to Female Extension Cable High Speed 5GBps Extension Cable Data Transfer for Keyboard, Mouse, Flash Drive, Hard Drive, Printer and More- 1.5M - Blue  https://www.amazon.in/Storite%C2%AE-150cm-Female-Extension-Printers/dp/B00OFM6PEO/ref=sr_1_500?qid=1672909149&s=electronics&sr=1-500

Top gợi ý không dùng category:


,product_id,product_name,category_path,rating,rating_count,img_link,product_link,similarity_score
92,B00NH11PEY,"AmazonBasics USB 2.0 - A-Male to A-Female Extension Cable for Personal Computer, Printer (Black, 9.8 Feet/3 Meters)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.5,74976.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41Fqm0bR7PL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-Extension-Cable-Male-Female/dp/B00NH11PEY/ref=sr_1_34?qid=1672909125&s=electronics&sr=1-34,0.411197
94,B00NH13Q8W,"AmazonBasics USB 2.0 Extension Cable for Personal Computer, Printer, 2-Pack - A-Male to A-Female - 3.3 Feet (1 Meter, Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.5,74977.0,https://m.media-amazon.com/images/I/41WuKPTQhTL._SY300_SX300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-Extension-Cable-2-Pack-Female/dp/B00NH13Q8W/ref=sr_1_233?qid=1672909135&s=electronics&sr=1-233,0.399643
93,B00NH12R1O,"Amazon Basics USB 3.0 Cable - A Male to Micro B - 6 Feet (1.8 Meters), Black",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.3,10911.0,https://m.media-amazon.com/images/I/41p9mn0fmIL._SY300_SX300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-USB-3-0-Cable-Meters/dp/B00NH12R1O/ref=sr_1_205?qid=1672909134&s=electronics&sr=1-205,0.321559
769,B08XXVXP3J,"Storite Super Speed USB 3.0 Male to Male Cable for Hard Drive Enclosures, Laptop Cooling Pad, DVD Players(60cm,Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.3,112.0,https://m.media-amazon.com/images/I/41XgWuRRNFL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Storite-USB-3-0-Transfer-Enclosures/dp/B08XXVXP3J/ref=sr_1_450?qid=1672909146&s=electronics&sr=1-450,0.302919
1110,B09YLYB9PB,"Ambrane 60W / 3A Fast Charging Output Cable with Micro to USB for Mobile, Neckband, True Wireless Earphone Charging, 480mbps Data Sync Speed, 1m Length (ACM - AZ1, Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.0,1423.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/414P4JCZY-L._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Ambrane-Charging-Neckband-Wireless-ACM/dp/B09YLYB9PB/ref=sr_1_192?qid=1672909133&s=electronics&sr=1-192,0.286483
855,B0994GFWBH,Lapster 1.5 mtr USB 2.0 Type A Male to USB A Male Cable for computer and laptop,Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.0,1313.0,https://m.media-amazon.com/images/I/310WOJIrwjL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Lapster-Type-Cable-computer-laptop/dp/B0994GFWBH/ref=sr_1_68?qid=1672909126&s=electronics&sr=1-68,0.285329
805,B094DQWV9B,"Kanget [2 Pack] Type C Female to USB A Male Charger | Charging Cable Adapter Converter compatible for iPhone 14, 13, 12,11 Pro Max/Mini/XR/XS/X/SE, Samsung S20 ultra/S21/S10/S8/S9/MacBook Pro iPad (Grey)",Computers&Accessories|Accessories&Peripherals|Adapters|USBtoUSBAdapters,4.0,1540.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/51JIngdPfEL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Kanget-Female-Adapter-Standard-Interface/dp/B094DQWV9B/ref=sr_1_171?qid=1672903004&s=computers&sr=1-171,0.284768
700,B08PPHFXG3,Posh 1.5 Meter High Speed Gold Plated HDMI Male to Female Extension Cable (Black),"Electronics|HomeTheater,TV&Video|Accessories|Cables|HDMICables",4.3,1237.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41-VkhORGAL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Meter-Speed-Plated-Female-Extension/dp/B08PPHFXG3/ref=sr_1_238?qid=1672909135&s=electronics&sr=1-238,0.276464
49,B00GE55L22,"Storite USB 3.0 Cable A to Micro B high Speed Upto 5 Gbps Data Transfer Cable for Portable External Hard Drive - (20cm), Black",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.1,2957.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images

# Recommender có cat

In [81]:
vectorizer_with_cat = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix_with_cat = vectorizer_with_cat.fit_transform(df["clean_text_with_category"])

print(tfidf_matrix_with_cat.shape)

(1351, 5000)


In [82]:
similarity_with_cat = cosine_similarity(tfidf_matrix_with_cat, tfidf_matrix_with_cat)
print(similarity_with_cat.shape)

(1351, 1351)


In [83]:
def recommend_with_category(product_id, top_k=10):
    if product_id not in product_idx_map:
        return pd.DataFrame()

    idx = product_idx_map[product_id]

    sim_scores = list(enumerate(similarity_with_cat[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_k+1]

    product_indices = [x[0] for x in sim_scores]
    scores = [x[1] for x in sim_scores]

    result = df.iloc[product_indices][[
        "product_id",
        "product_name",
        "category_path",
        "rating",
        "rating_count",
        "img_link",
        "product_link"
    ]].copy()

    result["similarity_score"] = scores
    return result

In [84]:
pd.set_option("display.max_columns", None)       # hiện tất cả cột
pd.set_option("display.max_colwidth", None)      # hiện đầy đủ nội dung trong ô
pd.set_option("display.width", None)             # tự mở rộng chiều rộng hiển thị
pd.set_option("display.max_rows", 100)           # hiện tối đa 100 dòng
pd.set_option("display.expand_frame_repr", False)

sample_id = df["product_id"].iloc[100]

print("Sản phẩm đầu vào:")
print(df[df["product_id"] == sample_id][["product_name","product_link"]])

print("\nTop gợi ý có dùng category:")
recommend_with_category(sample_id, top_k=10)

Sản phẩm đầu vào:
                                                                                                                                                                                      product_name                                                                                                                          product_link
100  Storite USB Extension Cable USB 3.0 Male to Female Extension Cable High Speed 5GBps Extension Cable Data Transfer for Keyboard, Mouse, Flash Drive, Hard Drive, Printer and More- 1.5M - Blue  https://www.amazon.in/Storite%C2%AE-150cm-Female-Extension-Printers/dp/B00OFM6PEO/ref=sr_1_500?qid=1672909149&s=electronics&sr=1-500

Top gợi ý có dùng category:


,product_id,product_name,category_path,rating,rating_count,img_link,product_link,similarity_score
92,B00NH11PEY,"AmazonBasics USB 2.0 - A-Male to A-Female Extension Cable for Personal Computer, Printer (Black, 9.8 Feet/3 Meters)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.5,74976.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41Fqm0bR7PL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-Extension-Cable-Male-Female/dp/B00NH11PEY/ref=sr_1_34?qid=1672909125&s=electronics&sr=1-34,0.437270
94,B00NH13Q8W,"AmazonBasics USB 2.0 Extension Cable for Personal Computer, Printer, 2-Pack - A-Male to A-Female - 3.3 Feet (1 Meter, Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.5,74977.0,https://m.media-amazon.com/images/I/41WuKPTQhTL._SY300_SX300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-Extension-Cable-2-Pack-Female/dp/B00NH13Q8W/ref=sr_1_233?qid=1672909135&s=electronics&sr=1-233,0.430628
93,B00NH12R1O,"Amazon Basics USB 3.0 Cable - A Male to Micro B - 6 Feet (1.8 Meters), Black",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.3,10911.0,https://m.media-amazon.com/images/I/41p9mn0fmIL._SY300_SX300_QL70_FMwebp_.jpg,https://www.amazon.in/AmazonBasics-USB-3-0-Cable-Meters/dp/B00NH12R1O/ref=sr_1_205?qid=1672909134&s=electronics&sr=1-205,0.352592
769,B08XXVXP3J,"Storite Super Speed USB 3.0 Male to Male Cable for Hard Drive Enclosures, Laptop Cooling Pad, DVD Players(60cm,Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.3,112.0,https://m.media-amazon.com/images/I/41XgWuRRNFL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Storite-USB-3-0-Transfer-Enclosures/dp/B08XXVXP3J/ref=sr_1_450?qid=1672909146&s=electronics&sr=1-450,0.338265
855,B0994GFWBH,Lapster 1.5 mtr USB 2.0 Type A Male to USB A Male Cable for computer and laptop,Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.0,1313.0,https://m.media-amazon.com/images/I/310WOJIrwjL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Lapster-Type-Cable-computer-laptop/dp/B0994GFWBH/ref=sr_1_68?qid=1672909126&s=electronics&sr=1-68,0.310949
1110,B09YLYB9PB,"Ambrane 60W / 3A Fast Charging Output Cable with Micro to USB for Mobile, Neckband, True Wireless Earphone Charging, 480mbps Data Sync Speed, 1m Length (ACM - AZ1, Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.0,1423.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/414P4JCZY-L._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Ambrane-Charging-Neckband-Wireless-ACM/dp/B09YLYB9PB/ref=sr_1_192?qid=1672909133&s=electronics&sr=1-192,0.307341
49,B00GE55L22,"Storite USB 3.0 Cable A to Micro B high Speed Upto 5 Gbps Data Transfer Cable for Portable External Hard Drive - (20cm), Black",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.1,2957.0,https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/31RK9+CyhoL._SY300_SX300_.jpg,https://www.amazon.in/Storite-USB-3-0-Micro-Cable/dp/B00GE55L22/ref=sr_1_201?qid=1672909134&s=electronics&sr=1-201,0.295042
1109,B09YLXYP7Y,"Ambrane 60W / 3A Fast Charging Output Cable with Type-C to USB for Mobile, Neckband, True Wireless Earphone Charging, 480mbps Data Sync Speed, 1m Length (ACT - AZ10, Black)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables,4.0,1423.0,https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/31l-eZHBfKL._SX300_SY300_QL70_FMwebp_.jpg,https://www.amazon.in/Ambrane-Charging-Neckband-Wireless-ACT/dp/B09YLXYP7Y/ref=sr_1_85?qid=1672909128&s=electronics&sr=1-85,0.289052
1108,B09YLX91QR,"Ambrane 60W / 3A Fast Charging Output Cable with Type-C to USB for Mobile, Neckband, True Wireless Earphone Charging, 480mbps Data Sync Speed, 1m Length (ACT - AZ10, White)",Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables

In [85]:
def same_category_at_k(product_id, recommender_fn, category_col="cat_level_2", top_k=5):
    # kiểm tra product_id tồn tại
    if product_id not in product_idx_map:
        return None
    
    # category của sản phẩm gốc
    source_category = df.loc[df["product_id"] == product_id, category_col].values[0]
    
    # lấy recommendations
    recs = recommender_fn(product_id, top_k=top_k)
    
    if recs.empty:
        return None
    
    # nối category từ df vào recs để so sánh
    recs_with_cat = recs.merge(
        df[["product_id", category_col]],
        on="product_id",
        how="left"
    )
    
    # đếm số item cùng category
    same_count = (recs_with_cat[category_col] == source_category).sum()
    
    return same_count / top_k

In [100]:
sample_id = df["product_id"].iloc[2]

score_no_cat = same_category_at_k(
    sample_id,
    recommend_no_category,
    category_col="cat_level_2",
    top_k=10
)

score_with_cat = same_category_at_k(
    sample_id,
    recommend_with_category,
    category_col="cat_level_2",
    top_k=10
)

print("Same-category@5 (no category):", score_no_cat)
print("Same-category@5 (with category):", score_with_cat)

Same-category@5 (no category): 0.9
Same-category@5 (with category): 0.9


In [101]:
def evaluate_same_category_all_products(recommender_fn, category_col="cat_level_2", top_k=5, sample_size=None):
    product_ids = df["product_id"].tolist()
    
    if sample_size is not None:
        product_ids = product_ids[:sample_size]
    
    scores = []
    
    for pid in product_ids:
        score = same_category_at_k(pid, recommender_fn, category_col=category_col, top_k=top_k)
        if score is not None:
            scores.append(score)
    
    if len(scores) == 0:
        return None
    
    return sum(scores) / len(scores)

In [102]:
same_cat5_no = evaluate_same_category_all_products(
    recommend_no_category,
    category_col="cat_level_2",
    top_k=5,
    sample_size=300
)

same_cat5_with = evaluate_same_category_all_products(
    recommend_with_category,
    category_col="cat_level_2",
    top_k=5,
    sample_size=300
)

print("Average Same-category@5 (no category):", same_cat5_no)
print("Average Same-category@5 (with category):", same_cat5_with)

Average Same-category@5 (no category): 0.8640000000000001
Average Same-category@5 (with category): 0.8886666666666669
